## Data Processing: JKP Global Factor Data (EM Universe)

Preprocesses the JKP EM parquet: coverage filter, per stock missing filter, cross sectional rank normalisation to [-0.5, 0.5], median imputation. Produces a firm lookup table with gvkey, country, and market equity for portfolio reporting. Computes per country firm counts and identifies qualifying countries for per country transformer training.

### 1. Setup

In [1]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

raw_path = Path('../data/Global Factor_EM.parquet')
output_dir = Path('../data/processed')
output_dir.mkdir(parents = True, exist_ok = True)

coverage_threshold = 0.70
max_miss_frac = 1.0 / 3.0
min_stocks = 30
min_firms_per_country = 50

train_end = '2014-12-31'
val_end = '2019-12-31'

# Identifiers and targets to load but not use as characteristics
load_always = ['id', 'gvkey', 'eom', 'excntry', 'ret_exc_lead1m', 'me']

# Columns to exclude from characteristic candidates
exclude_cols = {
	'id', 'gvkey', 'iid', 'permno', 'permco', 'date', 'eom', 'excntry',
	'size_grp', 'obs_main', 'exch_main', 'common', 'primary_sec',
	'source_crsp', 'comp_tpci', 'crsp_shrcd', 'comp_exchg', 'crsp_exchcd',
	'curcd', 'fx', 'adjfct', 'bidask',
	'ret', 'ret_local', 'ret_exc', 'ret_exc_lead1m',
	'prc', 'prc_local', 'prc_high', 'prc_low',
	'me', 'me_company', 'dolvol', 'shares', 'tvol',
	'ret_lag_dif', 'div_tot',
}

print(f'raw: {raw_path}')
print(f'output: {output_dir}')

raw: ..\data\Global Factor_EM.parquet
output: ..\data\processed


### 2. Load Raw Data and Apply JKP Screens

Read only the columns needed from the parquet. Sample screens (obs_main, exch_main, common, primary_sec) applied immediately.

In [2]:
schema = pq.read_schema(raw_path)
all_col_names = schema.names
print(f'total columns in raw parquet: {len(all_col_names)}')

screen_cols = [c for c in ['obs_main', 'exch_main', 'common', 'primary_sec'] if c in all_col_names]

char_candidate = [
	c for c in all_col_names
	if c not in exclude_cols
	and c not in screen_cols
	and c not in load_always
]
print(f'characteristic candidates: {len(char_candidate)}')

needed = load_always + screen_cols + char_candidate
needed = [c for c in needed if c in all_col_names]
print(f'loading {len(needed)} columns...')

df = pd.read_parquet(raw_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])
print(f'loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'date range: {df["eom"].min().date()} to {df["eom"].max().date()}')

# Downcast float64 to float32 immediately to halve memory
for col in char_candidate:
	if col in df.columns and df[col].dtype == np.float64:
		df[col] = df[col].astype(np.float32)
if 'me' in df.columns and df['me'].dtype == np.float64:
	df['me'] = df['me'].astype(np.float32)
gc.collect()
print(f'memory after float32 downcast: {df.memory_usage(deep = True).sum() / 1e9:.1f} GB')

# Apply JKP screens
n_before = len(df)
screen_mask = pd.Series(True, index = df.index)
for col in screen_cols:
	screen_mask = screen_mask & (df[col] == 1)

# Drop screen columns to free memory before selection
df = df.drop(columns = screen_cols, errors = 'ignore')
gc.collect()

# Single loc selection instead of chained indexing
keep_cols = [c for c in load_always + char_candidate if c in df.columns]
df = df.loc[screen_mask, keep_cols].reset_index(drop = True)
print(f'after screens: {len(df):,} rows ({len(df) / n_before * 100:.1f}%)')

char_candidate = [
	c for c in char_candidate
	if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]
print(f'numeric characteristic candidates: {len(char_candidate)}')
gc.collect()

total columns in raw parquet: 443
characteristic candidates: 407
loading 417 columns...
loaded: 4,386,856 rows x 417 columns
date range: 1995-01-31 to 2025-12-31
memory after float32 downcast: 9.1 GB
after screens: 4,386,856 rows (100.0%)
numeric characteristic candidates: 401


0

### 3. Coverage Filter

Select features with at least 70% non missing observations in the training period. Computed on training data only to prevent lookahead.

In [3]:
df_train_cov = df[df['eom'] <= train_end]
coverage = df_train_cov[char_candidate].notna().mean()
char_cols = sorted([c for c in char_candidate if coverage[c] >= coverage_threshold])
d = len(char_cols)
print(f'features with >= {coverage_threshold:.0%} coverage in training period: d = {d}')

retained_cov = coverage[char_cols]
print(f'coverage: min = {retained_cov.min():.3f} median = {retained_cov.median():.3f} max = {retained_cov.max():.3f}')

# Keep identifiers alongside characteristics
id_cols = [c for c in ['id', 'gvkey', 'eom', 'excntry', 'ret_exc_lead1m', 'me'] if c in df.columns]
df = df[id_cols + char_cols]
del df_train_cov
gc.collect()
print(f'working dataframe: {df.shape[0]:,} rows x {df.shape[1]} columns')

features with >= 70% coverage in training period: d = 151
coverage: min = 0.701 median = 0.777 max = 1.000
working dataframe: 4,386,856 rows x 157 columns


### 4. Monthly Preprocessing

For each month: drop firms missing more than one third of characteristics, rank normalise each characteristic cross sectionally to [-0.5, 0.5], replace remaining NaN with 0. Identifiers (id, gvkey, me, excntry) are carried through without transformation.

In [4]:
monthly_dates = sorted(df['eom'].unique())
print(f'processing {len(monthly_dates)} months...')

processed = []
skipped = 0

for i, eom in enumerate(monthly_dates):
	group = df[df['eom'] == eom]

	n_miss = group[char_cols].isna().sum(axis = 1)
	group = group[n_miss <= d * max_miss_frac]

	if len(group) < min_stocks:
		skipped += 1
		continue

	group = group.copy()
	ranks = group[char_cols].rank(pct = True, axis = 0) - 0.5
	ranks = ranks.fillna(0.0)
	group[char_cols] = ranks

	processed.append(group)

	if (i + 1) % 60 == 0:
		print(f'  {i + 1}/{len(monthly_dates)} months processed')

df_processed = pd.concat(processed, ignore_index = True)
print(f'done: {len(df_processed):,} rows, {df_processed["eom"].nunique()} months retained, {skipped} skipped')
del df, processed
gc.collect()

processing 372 months...
  60/372 months processed
  120/372 months processed
  180/372 months processed
  240/372 months processed
  300/372 months processed
  360/372 months processed
done: 3,758,613 rows, 372 months retained, 0 skipped


0

### 5. Country Analysis

Compute per country firm counts across months. Countries with at least the minimum average firm count qualify for per country transformer training. Smaller countries are excluded from the transformer comparison.

In [5]:
country_stats = (
	df_processed
	.groupby(['eom', 'excntry'])['id']
	.count()
	.reset_index()
	.rename(columns = {'id': 'n_firms'})
)

country_summary = (
	country_stats
	.groupby('excntry')['n_firms']
	.agg(['mean', 'min', 'max', 'count'])
	.rename(columns = {'count': 'n_months'})
	.sort_values('mean', ascending = False)
	.round(0)
	.astype(int)
)

qualifying_countries = sorted(
	country_summary[country_summary['mean'] >= min_firms_per_country].index.tolist()
)

print(f'Country firm counts (mean per month):')
print()
for country in country_summary.index:
	row = country_summary.loc[country]
	qualified = 'Y' if country in qualifying_countries else ' '
	print(
		f'  {country:5s}  mean: {row["mean"]:5.0f}  '
		f'min: {row["min"]:5.0f}  max: {row["max"]:5.0f}  '
		f'months: {row["n_months"]:4.0f}  [{qualified}]'
	)

print(f'\nQualifying countries (mean >= {min_firms_per_country}): {len(qualifying_countries)}')
print(f'  {qualifying_countries}')

excluded = [c for c in country_summary.index if c not in qualifying_countries]
print(f'Excluded ({len(excluded)}): {excluded}')

Country firm counts (mean per month):

  CHN    mean:  2124  min:    29  max:  5076  months:  372  [Y]
  IND    mean:  2009  min:     2  max:  4896  months:  372  [Y]
  TWN    mean:  1217  min:     4  max:  2145  months:  372  [Y]
  KOR    mean:   968  min:    64  max:  1989  months:  372  [Y]
  MYS    mean:   797  min:   171  max:  1036  months:  372  [Y]
  THA    mean:   491  min:    67  max:   882  months:  372  [Y]
  POL    mean:   416  min:     1  max:   802  months:  357  [Y]
  IDN    mean:   381  min:    30  max:   857  months:  372  [Y]
  TUR    mean:   270  min:     2  max:   579  months:  372  [Y]
  ZAF    mean:   246  min:    63  max:   306  months:  372  [Y]
  PHL    mean:   176  min:     8  max:   252  months:  372  [Y]
  GRC    mean:   153  min:     2  max:   248  months:  371  [Y]
  SAU    mean:   151  min:    11  max:   366  months:  290  [Y]
  BRA    mean:   127  min:     7  max:   253  months:  372  [Y]
  KWT    mean:   125  min:     4  max:   174  months:  297  [Y]
 

### 6. Firm Lookup Table

Maps each firm id to its gvkey, country, and latest available market equity. Used by the transformer notebook to identify companies in the portfolio holdings.

In [6]:
# Build firm lookup: one row per firm with latest market equity
has_me = 'me' in df_processed.columns
has_gvkey = 'gvkey' in df_processed.columns

if has_me:
	# Use the latest non-null market equity for each firm
	me_latest = (
		df_processed[df_processed['me'].notna()]
		.sort_values('eom')
		.groupby('id')
		.last()[['me']]
		.reset_index()
	)
else:
	me_latest = pd.DataFrame({'id': df_processed['id'].unique()})
	me_latest['me'] = np.nan

# One row per firm with country and gvkey
firm_info_cols = ['id', 'excntry']
if has_gvkey:
	firm_info_cols.append('gvkey')
firm_info = (
	df_processed[firm_info_cols]
	.drop_duplicates(subset = ['id'])
)

firm_lookup = firm_info.merge(me_latest, on = 'id', how = 'left')
firm_lookup = firm_lookup.sort_values('id').reset_index(drop = True)

print(f'firm lookup: {len(firm_lookup):,} unique firms')
print(f'columns: {list(firm_lookup.columns)}')
print(f'countries: {firm_lookup["excntry"].nunique()}')
if has_me:
	print(f'market equity coverage: {firm_lookup["me"].notna().mean():.1%}')
print()
print(firm_lookup.head(10))

firm lookup: 26,313 unique firms
columns: ['id', 'excntry', 'gvkey', 'me']
countries: 24
market equity coverage: 100.0%

            id excntry   gvkey           me
0  300185503.0     PHL  001855   364.649475
1  300216202.0     PHL  002162    36.489510
2  300704101.0     MYS  007041    69.022781
3  300830301.0     ZAF  008303   495.138824
4  300854401.0     PHL  008544  4624.893066
5  301075501.0     MEX  010755   554.273560
6  301551001.0     ZAF  015510  1751.064819
7  301560402.0     ZAF  015604  1643.564209
8  301561801.0     ZAF  015618  7675.139160
9  301562101.0     ZAF  015621   930.677490


### 7. Train / Val / Test Split

In [ ]:
train_mask = df_processed['eom'] <= train_end
val_mask = (df_processed['eom'] > train_end) & (df_processed['eom'] <= val_end)
test_mask = df_processed['eom'] > val_end

df_train = df_processed[train_mask].copy()
df_val = df_processed[val_mask].copy()
df_test = df_processed[test_mask].copy()

print(f'train:{len(df_train):,} rows, {df_train["eom"].nunique()} months '
	  f'({df_train["eom"].min().date()} to {df_train["eom"].max().date()})')
print(f'val:{len(df_val):,} rows, {df_val["eom"].nunique()} months '
	  f'({df_val["eom"].min().date()} to {df_val["eom"].max().date()})')
print(f'test:{len(df_test):,} rows, {df_test["eom"].nunique()} months '
	  f'({df_test["eom"].min().date()} to {df_test["eom"].max().date()})')

train: 1,553,371 rows, 240 months (1995-01-31 to 2014-12-31)
val:   892,993 rows, 60 months (2015-01-31 to 2019-12-31)
test:  1,312,249 rows, 72 months (2020-01-31 to 2025-12-31)


### 8. Save

In [8]:
df_train.to_parquet(output_dir / 'train.parquet', index = False)
df_val.to_parquet(output_dir / 'val.parquet', index = False)
df_test.to_parquet(output_dir / 'test.parquet', index = False)
print(f'saved train/val/test parquets')

firm_lookup.to_parquet(output_dir / 'firm_lookup.parquet', index = False)
print(f'saved firm_lookup.parquet ({len(firm_lookup):,} firms)')

country_summary.to_csv(output_dir / 'country_stats.csv')
print(f'saved country_stats.csv')

country_codes = sorted(df_processed['excntry'].dropna().unique().tolist())
country_to_id = {c: i for i, c in enumerate(country_codes)}

metadata = {
	'char_cols': char_cols,
	'd': d,
	'coverage_threshold': coverage_threshold,
	'train_end': train_end,
	'val_end': val_end,
	'country_to_id': country_to_id,
	'country_codes': country_codes,
	'qualifying_countries': qualifying_countries,
	'min_firms_per_country': min_firms_per_country,
}
with open(output_dir / 'feature_metadata.json', 'w') as f:
	json.dump(metadata, f, indent = 2)
print(f'saved feature_metadata.json (d = {d}, {len(qualifying_countries)} qualifying countries)')

saved train/val/test parquets
saved firm_lookup.parquet (26,313 firms)
saved country_stats.csv
saved feature_metadata.json (d = 151, 20 qualifying countries)


### 9. Sanity Checks

In [ ]:
sample_eom = df_train['eom'].iloc[100]
sample = df_train[df_train['eom'] == sample_eom][char_cols]
print(f'sample month {sample_eom.date()}: {len(sample)} firms')
print(f'feature range: [{sample.min().min():.4f}, {sample.max().max():.4f}] (expect approx [-0.5, 0.5])')
print(f'feature mean:{sample.mean().mean():.6f} (expect approx 0)')
print(f'nan count:{sample.isna().sum().sum()} (expect 0)')

ret_miss = df_train['ret_exc_lead1m'].isna().mean()
print(f'ret_exc_lead1m missing rate in train: {ret_miss:.2%}')

monthly_n = df_processed.groupby('eom').size()
print(f'monthly cross section: min = {monthly_n.min()}, mean = {monthly_n.mean():.0f}, max = {monthly_n.max()}')

# Verify qualifying countries have sufficient firms
print(f'\nQualifying country cross section sizes (test period):')
test_stats = (
	df_test.groupby('excntry')['id'].count()
	.reset_index()
	.rename(columns = {'id': 'total_obs'})
)
test_stats['months'] = df_test['eom'].nunique()
test_stats['mean_per_month'] = (test_stats['total_obs'] / test_stats['months']).round(0).astype(int)
for _, row in test_stats.sort_values('mean_per_month', ascending = False).iterrows():
	q = 'Y' if row['excntry'] in qualifying_countries else ' '
	print(f'{row["excntry"]:5s}  {row["mean_per_month"]:5d} firms/month  [{q}]')

del df_train, df_val, df_test, df_processed
gc.collect()

sample month 1995-01-31: 518 firms
feature range: [-0.4981, 0.5000] (expect approx [-0.5, 0.5])
feature mean:  0.000965 (expect approx 0)
nan count:     0 (expect 0)
ret_exc_lead1m missing rate in train: 1.02%
monthly cross section: min = 517, mean = 10104, max = 20044

Qualifying country cross section sizes (test period):
  CHN     4625 firms/month  [Y]
  IND     4026 firms/month  [Y]
  TWN     2044 firms/month  [Y]
  KOR     1818 firms/month  [Y]
  MYS      950 firms/month  [Y]
  THA      813 firms/month  [Y]
  IDN      744 firms/month  [Y]
  POL      652 firms/month  [Y]
  TUR      478 firms/month  [Y]
  SAU      262 firms/month  [Y]
  PHL      244 firms/month  [Y]
  BRA      233 firms/month  [Y]
  ZAF      231 firms/month  [Y]
  EGY      195 firms/month  [Y]
  CHL      154 firms/month  [Y]
  KWT      143 firms/month  [Y]
  GRC      139 firms/month  [Y]
  ARE      116 firms/month  [Y]
  MEX      113 firms/month  [Y]
  PER       96 firms/month  [Y]
  QAT       50 firms/month  [ ]
  C